In [ ]:
!uv pip install rouge_score evaluate

- Initialization: Start with N=5 candidate prompts 
- Evaluation: For each candidate, generate new outputs and compare them to the original outputs.
- Selection & Mutation: 
	- The best candidate is chosen based on ROUGE-1 scoring, which measures word overlap between outputs generated by the candidate prompt and the original outputs.
	- Modify the candidate prompts based on the differences identified.
- Iteration: Repeat the process for X iterations until the reconstructed prompt best approximates the hidden prompt

In [ ]:
import re
import numpy as np
import evaluate
from tqdm.notebook import trange, tqdm
import pandas as pd
import json
from langchain_ollama import ChatOllama

class RPE():
    def __init__(self, X=1, N=1, model=None, p=0.5):
        '''
        Initialize the optimization process
        parameter:
          X: number of iteration for optimization
          N: number of candidate prompts to keep at each optimization step
          p: probability to add words from answers to children (mutation probability) #not using          
        '''
        self.X = X
        self.N = N
        self.p = p
        self.llms =  [{str(t): ChatOllama(model=model,reasoning=False,temperature=t,num_predict=1024)} for t in [0,0.5,1]]        
        self.answers = []
        self.model = model
        self.candidate_prompts = []
        self.candidate_scores = []
        self.candidate_total_scores = []
        self.candidate_answers = []
        self.parent_candidates = []        
        self.ori_prompt = None
        self.answers_word_set = None
        self.rouge = evaluate.load('rouge')

    def reset(self, X=1, N=1, p=0.5):
        self.X = X
        self.N = N
        self.p = p
        self.answers = []
        
        self.candidate_prompts = []
        self.candidate_scores = []
        self.candidate_total_scores = []
        self.candidate_answers = []
        self.parent_candidates = []
        self.answers_word_set = None
        
    def __call__(self, ini_prompts=None, ini_scores=None, ini_total_scores=None, ini_answers=None):
        """
        Recover prompt from answers
        """
        #generate prompt based on answers
        self.initial_prompt_generation(ini_prompts)
        #evaluation
        if ini_scores and ini_total_scores and ini_answers:
            self.candidate_scores, self.candidate_total_scores, self.candidate_answers = ini_scores, ini_total_scores, ini_answers
        else:
            self.candidate_scores, self.candidate_total_scores, self.candidate_answers = self.evaluation(self.candidate_prompts)

        
        for i in range(self.X):
            #crossover
            children_candidates = self.cross_mutation()

            #evaluation
            
            children_scores, children_total_scores, children_answers = self.evaluation(children_candidates)

            #replacement
            self.replacement(children_candidates, children_scores, children_answers, children_total_scores)

        idx = np.argmax(self.candidate_scores)
        final_prompt = self.candidate_prompts[idx]
        return final_prompt
        


    def chat(self, messages, n=1, t=0.5):
        """
        define chat function
        """
        _llm = self.llms[str(t)][str(t)]
        text_response = []
        for i in range(n):
            rs = _llm.invoke(messages)
            text_response.append(rs.content)        
        return text_response

    def generate_answers(self, prompt=None, answers:list=[], n:int=1, t=0.5):
        """
        Generate answers we want to infer from. Create tag dict for all answers
        parameter:
          prompt: the original prompt we use to generate answers
          answers: optional, if provided, just copy this list to self.answers
          n: number of answers to generate for searching original prompt
        """
        if answers:
            self.answers = answers
        else:
            message = [{"role":"user", "content":prompt}]
            self.answers = self.chat(message, n=n, t=t)
        
        self.ori_prompt = prompt
        

    def initial_prompt_generation(self, ini_prompts=None):
        if ini_prompts:
            for p in ini_prompts:
                self.candidate_prompts.append(p)
        else:
            prompt_0 = "All answers above are generated by one prompt passing through LLM multiple times. Based on the answers provided above, speculate the underlying prompt."
            message_0 = [{"role":"system", "content":"As a smart dectactive, you can infer the underlying prompt besed on outputs of language model."}] + \
                    [{"role":"user", "content":"Answer "+str(i)+": "+self.answers[i]} for i in range(len(self.answers))] + \
                    [{"role":"user", "content":prompt_0}] + \
                    [{"role":"system", "content":"Think step by step and return the prompt beginning with <PRO> and end with </PRO>."}]
            prompt_candidates = self.chat(message_0, n=self.N, t=1)
            for prompt in prompt_candidates:
                splited_prompt = re.split('<PRO>|</PRO>', prompt)
                #if the format is not correct, regenerate the prompt until the format is correct
                while len(splited_prompt) != 3:
                    prompt = self.chat(message_0, n=1)[0]
                    splited_prompt = re.split('<PRO>|</PRO>', prompt)
                self.candidate_prompts.append(splited_prompt[1])

    def evaluation(self, prompt_list = []):
        """
        Evaluate each pompt candidate in prompt_list
        """
        total_score_list = []
        final_score_list = []
        candidate_answers_list = []
        n = len(prompt_list)
        m = len(self.answers)
        for j in range(n):
            
            message_prompt = [{"role":"user", "content":prompt_list[j]}]
            answer_e = self.chat(message_prompt, t=0)[0]
            score_list = []
            
            for i in range(m):
                
                predictions = [answer_e]
                references = [self.answers[i]]
                results = self.rouge.compute(predictions=predictions,
                          references=references)
                score = results["rouge1"]
                score_list.append(float(score))
                
                
               
                
            final_score = (np.mean(score_list)+max(score_list))/2
            #final_score = np.mean(score_list)
            #final_score = max(score_list)
            total_score_list.append(score_list)
            final_score_list.append(final_score)
            candidate_answers_list.append(answer_e)
            
        return final_score_list, total_score_list, candidate_answers_list


    def cross_mutation(self):
        '''
        For each candidate prompt, get difference between candidate answer and original answers
        summarize that
        ask llm to give advise of revising prompt
        do the revise
        '''
        children_candidates = []
        
        for i in range(5):
            old_prompt = self.candidate_prompts[i]
            old_answer = self.candidate_answers[i]
            m = len(self.answers)
            suggest_list = []
            #difference between answers
            for i in range(m):
                prompt_e = "You should answer concisely with no redundant information. Given two answers below, how is the candidate answer different from "+\
                "the reference answer? If there is no significant difference, you can answer 'There is no difference'."
                message_e = [{"role":"user", "content": prompt_e}]+\
                            [{"role":"user", "content": "Candidate Answer: "+old_answer}]+\
                            [{"role":"user", "content": "Reference Answer: "+self.answers[i]}]
                suggest_e = self.chat(message_e)
                suggest_list.append(suggest_e[0])

            #summarize difference
            prompt_s = "Given a list of responses, summarize them into one concise response."
            message_s = [{"role":"user", "content": prompt_s}] + \
                        [{"role":"user", "content": "List of responses: " + str(suggest_list)}]
            summary_s = self.chat(message_s)[0]

            #generate advice to revise prompt
            prompt_p = "According to the difference above, how should you change the prompt that "+\
                        "generate the candidate answer to make the candidate answer more similar to the reference answer? Answer concisely."
            message_p = [{"role":"user", "content": "Difference: " + summary_s}] + \
                        [{"role":"user", "content": prompt_p}] 
            difference_p = self.chat(message_p)[0]

            #revise the prompt
            prompt_r = "Using the suggestion above to revise the following prompt." 
            message_r = [{"role":"user", "content": "Suggestion: " + difference_p}] + \
                        [{"role":"user", "content": prompt_r}] + \
                        [{"role":"user", "content": "Prompt: " + old_prompt}] + \
                        [{"role":"user", "content": "The prompt should be concise. Return the prompt beginning with <PRO> and end with </PRO>."}]
            child_r = self.chat(message_r)[0]

            splited_child = re.split('<PRO>|</PRO>', child_r)
            while len(splited_child) != 3:
                child_r = self.chat(message_r)[0]
                splited_child = re.split('<PRO>|</PRO>', child_r)
            children_candidates.append(splited_child[1])
            
        return children_candidates
        
    def replacement(self, children_prompts, children_scores, children_answers, children_total_scores):
        for i in range(5):
            child_score = children_scores[i]
            if child_score > min(self.candidate_scores):
                idx = np.argmin(self.candidate_scores)
                self.candidate_prompts[idx] = children_prompts[i]
                self.candidate_answers[idx] = children_answers[i]
                self.candidate_total_scores[idx] = children_total_scores[i]
                self.candidate_scores[idx] = child_score

In [ ]:
#read dataset
#data_prompt = pd.read_csv("./tmp/model-inversion-prompt.csv", index_col=0)
#list_prompt = list(data_prompt["prompt"][:1])
list_prompt = ["Generate a report summarizing the key statistics from the given dataset. https://data.worldbank.org/indicator/SP.POP.TOTL"]

#record initial answers
ori_answers = []
#record final answer
final_outputs = []
#initialize optimizer
rpe = RPE(X=5, N=5, model="gemma3:1b")
n = len(list_prompt)
print("Algorithm starts")
for i in trange(n):
    p = list_prompt[i]
    rpe.reset(X=5, N=5)
    #o_answers = ori_answers[i]
    rpe.generate_answers(prompt=p, answers=[], n=5)
    #rpe.generate_answers(prompt=p, answers=o_answers, n=5)
    output = rpe(ini_prompts=None, ini_scores=None, ini_total_scores=None, ini_answers=None)
    final_outputs.append(output)
    ori_answers.append(rpe.answers)
    
#save final prompt and initial answers
with open('./tmp/rpe_prompts.json', 'w') as outfile:
    json.dump(final_outputs, outfile)
with open('./tmp/rpe_answers.json', 'w') as outfile:
    json.dump(ori_answers, outfile)

Algorithm starts


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#exec time: 21m40s x prompt